In [2]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week8-assignment-7"). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

## pivot table

In [3]:
schema1 = 'loglevel string , logtime string'

In [4]:
logdf = spark.read.format("csv").schema(schema1).load("/public/trendytech/datasets/logdata1m.csv")

In [5]:
logdf1 = logdf.withColumn("logtime",to_timestamp("logtime"))  ## convert string to timestamp

In [6]:
logdf1.createOrReplaceTempView("serverlogs")

In [7]:
spark.sql("select loglevel, date_format(logtime,'MMM') as month from serverlogs").show(10)

+--------+-----+
|loglevel|month|
+--------+-----+
|    INFO|  Aug|
|    WARN|  Jan|
|    INFO|  Jun|
|    INFO|  Jan|
|   DEBUG|  Jul|
|    INFO|  Feb|
|    INFO|  Jul|
|    INFO|  Apr|
|   DEBUG|  Nov|
|    INFO|  Aug|
+--------+-----+
only showing top 10 rows



## PIVOT TABLE

#### .groupBy("").pivot("").count()

In [9]:
spark.sql("select loglevel, date_format(logtime,'MMMM') as month from serverlogs") \
.groupBy("loglevel").pivot("month").count().show()

+--------+-----+------+--------+--------+-------+-----+-----+-----+-----+--------+-------+---------+
|loglevel|April|August|December|February|January| July| June|March|  May|November|October|September|
+--------+-----+------+--------+--------+-------+-----+-----+-----+-----+--------+-------+---------+
|    INFO|29302| 28993|   28874|   28983|  29119|29300|29143|29095|28900|   23301|  29018|    29038|
|   ERROR| 4107|  3987|    4106|    4013|   4054| 3976| 4059| 4122| 4086|    3389|   4040|     4161|
|    WARN| 8277|  8381|    8328|    8266|   8217| 8222| 8191| 8165| 8403|    6616|   8226|     8352|
|   DEBUG|41869| 42147|   41749|   41734|  41961|42085|41774|41652|41785|   33366|  41936|    41433|
|   FATAL|   83|    80|      94|      72|     94|   98|   78|   70|   60|   16797|     92|       81|
+--------+-----+------+--------+--------+-------+-----+-----+-----+-----+--------+-------+---------+



In [11]:
spark.sql("select loglevel,int( date_format(logtime,'M')) as month from serverlogs") \
.groupBy("loglevel").pivot("month").count().show()

+--------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|loglevel|    1|    2|    3|    4|    5|    6|    7|    8|    9|   10|   11|   12|
+--------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|    INFO|29119|28983|29095|29302|28900|29143|29300|28993|29038|29018|23301|28874|
|   ERROR| 4054| 4013| 4122| 4107| 4086| 4059| 3976| 3987| 4161| 4040| 3389| 4106|
|    WARN| 8217| 8266| 8165| 8277| 8403| 8191| 8222| 8381| 8352| 8226| 6616| 8328|
|   FATAL|   94|   72|   70|   83|   60|   78|   98|   80|   81|   92|16797|   94|
|   DEBUG|41961|41734|41652|41869|41785|41774|42085|42147|41433|41936|33366|41749|
+--------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+



In [20]:
month_list = ['January','February','March','April','May','June','July','August','September','October','November','December','Something']

### save some processing by listing the pivot columns rather than letting spark find them out

In [21]:
spark.sql("select loglevel,date_format(logtime,'MMMM') as month from serverlogs") \
.groupBy("loglevel").pivot("month",month_list).count().show()

+--------+-------+--------+-----+-----+-----+-----+-----+------+---------+-------+--------+--------+---------+
|loglevel|January|February|March|April|  May| June| July|August|September|October|November|December|Something|
+--------+-------+--------+-----+-----+-----+-----+-----+------+---------+-------+--------+--------+---------+
|    INFO|  29119|   28983|29095|29302|28900|29143|29300| 28993|    29038|  29018|   23301|   28874|     null|
|   ERROR|   4054|    4013| 4122| 4107| 4086| 4059| 3976|  3987|     4161|   4040|    3389|    4106|     null|
|    WARN|   8217|    8266| 8165| 8277| 8403| 8191| 8222|  8381|     8352|   8226|    6616|    8328|     null|
|   FATAL|     94|      72|   70|   83|   60|   78|   98|    80|       81|     92|   16797|      94|     null|
|   DEBUG|  41961|   41734|41652|41869|41785|41774|42085| 42147|    41433|  41936|   33366|   41749|     null|
+--------+-------+--------+-----+-----+-----+-----+-----+------+---------+-------+--------+--------+---------+



In [ ]:
#   pivot() without values triggers an extra distinct-scan job to discover columns and orders them alphabetically; 
#     passing the value list explicitly skips that scan, 
#     preserves your intended column order, and guarantees a stable schema — 
#     which is why it's the production-recommended form.

In [22]:
## if we dont give month_list, if a month had no logs, that month will be skipped in the pivot table as it has no rows. but if we provide month_list, 
## all the months in the list will be listed in pivot table even if no rows